# FabricCare V3 · legacy 74-feature GLCM experiment

Data source is [FabriCare_Dataset_V3](https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8). Add its shortcut directly to **My Drive** under its published name. The setup verifies the V3 split manifest and stages its raw files locally. Run cells in order.

This feature experiment has its own export format. Use the MobileNetV2 Colab notebook for the app-compatible 72-feature Decision Tree and SVM exports.


In [ ]:
# Pin scikit-learn to match Fabricare's model loader.
%pip -q install scikit-learn==1.6.1 scikit-image==0.26.0 opencv-python-headless==4.12.0.88 numpy==2.2.6 joblib==1.5.3

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import json, hashlib, platform, shutil
import cv2
import numpy as np
import sklearn, skimage, joblib
from skimage.feature import graycomatrix, graycoprops
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tempfile import mkdtemp
drive.mount('/content/drive')

import csv
import hashlib
import io
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps

V3_URL = "https://drive.google.com/drive/folders/1cHixSkWAvL3gK1Mhuw8s6G0E7RMYx0-8"
V3_ROOT = Path("/content/drive/MyDrive/FabriCare_Dataset_V3")
MANIFEST_SHA256 = "f164042dcb4a0b4876fc5474532df1bb7b25403703e4a70852f19ada64c287ea"
SPLITS = ("train", "valid", "test")
CLASS_NAMES = ["cotton", "denim", "linen", "polyester", "silk", "wool"]
SEED = 42

def read_v3_manifest(root):
    manifest = root / "reports" / "split_manifest_70_20_10.csv"
    if not manifest.is_file():
        raise FileNotFoundError(f"Open {V3_URL} and add its shortcut to My Drive as {root.name}.")
    data = manifest.read_bytes()
    if hashlib.sha256(data).hexdigest() != MANIFEST_SHA256:
        raise ValueError("The split manifest differs from the audited V3 revision. Verify the folder and split file.")
    rows = list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"), newline="")))
    counts, groups, digests = Counter(), defaultdict(set), set()
    for row in rows:
        split, label, filename = row["split"], row["fabric"], row["filename"]
        if split not in SPLITS or label not in CLASS_NAMES or not filename or Path(filename).name != filename:
            raise ValueError(f"Invalid V3 manifest entry: {row}")
        digest = row["sha256"]
        if digest in digests:
            raise ValueError(f"Duplicate content SHA-256 in the split manifest: {filename}")
        if not row["group_id"]:
            raise ValueError(f"Missing garment group: {filename}")
        digests.add(digest)
        groups[row["group_id"]].add(split)
        counts[split, label] += 1
    if any(len(value) > 1 for value in groups.values()):
        raise ValueError("A garment group appears in multiple splits.")
    for split in SPLITS:
        if any(counts[split, label] == 0 for label in CLASS_NAMES):
            raise ValueError(f"Missing fabric class in {split}.")
        print(split, {label: counts[split, label] for label in CLASS_NAMES})
    return rows

def stage_v3(root, records, target, *, padded):
    staged = []
    for record in records:
        label, split = record["fabric"], record["split"]
        source = root / "raw" / label / record["filename"]
        content = source.read_bytes()
        if hashlib.sha256(content).hexdigest() != record["sha256"]:
            raise ValueError(f"Image bytes differ from the V3 manifest: {source}")
        parent = target / split / label
        parent.mkdir(parents=True, exist_ok=True)
        if padded:
            destination = parent / (record["sha256"] + ".png")
            with Image.open(io.BytesIO(content)) as raw:
                # Normalize archival originals before submitting them to the classifier.
                rgb = ImageOps.exif_transpose(raw).convert("RGB")
                ImageOps.pad(rgb, (224, 224), method=Image.Resampling.BILINEAR,
                             color=(127, 127, 127)).save(destination)
        else:
            destination = parent / record["filename"]
            destination.write_bytes(content)
        staged.append((str(destination), CLASS_NAMES.index(label)))
    return staged

if sklearn.__version__ != '1.6.1':
    raise RuntimeError('Restart the runtime after package installation.')
DATASET_DIR = Path(mkdtemp(prefix='fabricare-v3-raw-', dir='/content'))
stage_v3(V3_ROOT, read_v3_manifest(V3_ROOT), DATASET_DIR, padded=False)
OUTPUT_DIR = Path('/content/drive/MyDrive/Fabricare_training_v3') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Exports:', OUTPUT_DIR)


## Extract exactly the same features as the API

Uses 74 GLCM features matching the API. Identical same-label images are retained once, preferring train, valid, then test. Drive source images are untouched. Conflicting labels, unreadable images and classes emptied by filtering stop training. Missing evaluation classes are explicitly reported as untested. Cached features are reused only in the same session; restart the session if source images change. Different burst shots/crops still need to be grouped by physical garment.


In [ ]:
# Reuse the feature snapshot from this session when available.
# Restart the session if source images have changed since extraction.
_feature_cache = {}
if 'data' in globals() and 'manifest' in globals():
    for _split, (_x, _y) in data.items():
        _records = [r for r in manifest if r['split'] == _split]
        if len(_records) == len(_x) == len(_y):
            for _r, _v, _label in zip(_records, _x, _y):
                if _r['label'] == str(_label) and np.asarray(_v).shape == (74,):
                    _feature_cache[_r['path']] = (_r['decoded_sha256'], np.asarray(_v).copy())
print('Reusable feature rows:', len(_feature_cache))

def extract_glcm_features(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'Cannot read image: {image_path}')
    image = cv2.resize(image, (128, 128))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = (gray / 32).astype(np.uint8)
    glcm = graycomatrix(gray, distances=[1, 2, 4],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=8, symmetric=True, normed=True)
    features = []
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']:
        features.extend(graycoprops(glcm, prop).flatten())
    features.extend([np.mean(gray), np.std(gray)])
    return np.array(features).reshape(1, -1)

# Keep one copy per decoded image, preferring train, then valid, then test.
# Source images are never edited, moved, or deleted.
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
data, counts, source_counts, manifest, seen = {}, {}, {}, [], {}
duplicate_report, errors = [], []
for split in ['train', 'valid', 'test']:
    split_dir = DATASET_DIR / split
    if not split_dir.is_dir():
        raise ValueError(f'Missing split: {split_dir}')
    actual_classes = {p.name for p in split_dir.iterdir() if p.is_dir() and not p.name.startswith('.')}
    if actual_classes != set(CLASS_NAMES):
        raise ValueError(f'{split}: expected {CLASS_NAMES}, found {sorted(actual_classes)}')
    features, labels = [], []
    counts[split], source_counts[split] = {}, {}
    for label in CLASS_NAMES:
        paths = sorted(p for p in (split_dir / label).rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
        source_counts[split][label] = len(paths)
        counts[split][label] = 0
        for path in paths:
            relative_path = str(path.relative_to(DATASET_DIR))
            cached = _feature_cache.get(relative_path)
            if cached is not None:
                digest, vector = cached
            else:
                image = cv2.imread(str(path))
                if image is None:
                    errors.append(f'Unreadable: {path}')
                    continue
                digest = hashlib.sha256(str(image.shape).encode() + image.tobytes()).hexdigest()
                vector = extract_glcm_features(path)[0]
            previous = seen.get(digest)
            if previous:
                if previous['label'] != label:
                    errors.append(f'Conflicting labels for identical image: {previous["path"]} and {relative_path}')
                else:
                    duplicate_report.append({'skipped_path': relative_path, 'skipped_split': split,
                        'kept_path': previous['path'], 'kept_split': previous['split'],
                        'label': label, 'decoded_sha256': digest})
                continue
            record = {'path': relative_path, 'split': split, 'label': label, 'decoded_sha256': digest}
            seen[digest] = record
            features.append(vector)
            labels.append(label)
            manifest.append(record)
            counts[split][label] += 1
        if counts[split][label] == 0:
            if split == 'train' or source_counts[split][label] > 0:
                errors.append(f'No unique readable images remain: {split}/{label}. Add independent photos to this split.')
            else:
                print(f'WARNING: {split}/{label} has no images. This class cannot be evaluated in this split.')
    if not features:
        errors.append(f'No images retained in {split}')
    data[split] = (np.asarray(features), np.asarray(labels))
    print(split, 'source:', source_counts[split], 'retained:', counts[split])
print(f'Skipped {len(duplicate_report)} identical same-label copies. No Drive images were modified.')
for item in duplicate_report[:10]:
    print('Skipped:', item['skipped_path'], '| Kept:', item['kept_path'])
if errors:
    print('\n'.join(errors[:30]))
    raise ValueError(f'{len(errors)} unresolved dataset issue(s). Review conflicting labels, unreadable files, or empty classes before training.')
assert all(x.shape[1] == 74 for x, y in data.values())
print('Exact-image duplicate checks passed. Remaining burst shots/crops of the same garment must still stay in one split.')
print('Validation and test results apply to the retained unique images, not the original unfiltered splits.')


## Train and choose settings using validation data
The test split remains excluded from model fitting and parameter selection. Selection uses macro F1 so every fabric contributes equally. Decision Tree probabilities are model scores, not a guarantee of correctness.

In [ ]:
X_train, y_train = data['train']
X_valid, y_valid = data['valid']
trials = []
best_model, best_score = None, -1
for depth in [3, 5, 8, 12, None]:
    for leaf in [2, 5, 10]:
        candidate = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
            class_weight='balanced', random_state=SEED)
        candidate.fit(X_train, y_train)
        predicted = candidate.predict(X_valid)
        score = f1_score(y_valid, predicted, labels=CLASS_NAMES, average='macro', zero_division=0)
        trials.append({'max_depth': depth, 'min_samples_leaf': leaf, 'validation_macro_f1': float(score)})
        if score > best_score:
            best_model, best_score = candidate, score
print('Selected:', best_model)
print('Validation macro F1:', round(best_score, 4))
print(classification_report(y_valid, best_model.predict(X_valid), labels=CLASS_NAMES, zero_division=0))

In [ ]:
# Final held-out evaluation. Do not repeatedly tune against these results.
X_test, y_test = data['test']
y_pred = best_model.predict(X_test)
metrics = {
    'test_accuracy': float(accuracy_score(y_test, y_pred)),
    'test_macro_f1': float(f1_score(y_test, y_pred, labels=CLASS_NAMES, average='macro', zero_division=0)),
    'per_class': classification_report(y_test, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0),
    'confusion_matrix_label_order': CLASS_NAMES,
    'confusion_matrix': confusion_matrix(y_test, y_pred, labels=CLASS_NAMES).tolist(),
}
print('Test accuracy:', round(metrics['test_accuracy'], 4))
print('Test macro F1:', round(metrics['test_macro_f1'], 4))
print(classification_report(y_test, y_pred, labels=CLASS_NAMES, zero_division=0))
print('Confusion matrix (rows = actual, columns = predicted):', CLASS_NAMES)
print(np.array(metrics['confusion_matrix']))

## Try a new phone photo
Upload a clear close-up of fabric texture, similar in framing to your training images. The model always chooses among its known classes; it cannot reliably reject unknown materials or determine blend percentages. An uncertain result requires review. This cell does not change model training.

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename, content in uploaded.items():
    path = Path('/content') / ('fabricare_phone_' + Path(filename).name)
    path.write_bytes(content)
    features = extract_glcm_features(path)
    scores = best_model.predict_proba(features)[0]
    ranked = sorted(zip(best_model.classes_, scores), key=lambda item: item[1], reverse=True)
    print(filename, '->', ranked[0][0])
    print({str(label): round(float(score) * 100, 2) for label, score in ranked})

## Export for Fabricare
Only replace your backend model after reviewing evaluation and real capture results. Back up existing files first. Copy both `fabric_glcm_decision_tree.pkl` and `fabric_labels.json` into Fabricare's `model/` directory and restart the Python API. The backend already accepts string class names. Keep scikit-learn 1.6.1 in the backend. Never load model pickle files from untrusted sources.

This exports the selected model trained on `train`, without silently retraining on validation or test images. Your damage detector `best.pt` remains separate. Training here is a baseline; measured results should guide any later architecture changes.

In [ ]:
from google.colab import files
OUTPUT_DIR = Path('/content/drive/MyDrive/Fabricare_training_v3') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
model_path = OUTPUT_DIR / 'fabric_glcm_decision_tree.pkl'
joblib.dump(best_model, model_path)
(OUTPUT_DIR / 'fabric_labels.json').write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
metadata = {
    'dataset': str(DATASET_DIR), 'seed': SEED, 'counts': counts,
    'source_counts': source_counts, 'skipped_identical_copies': len(duplicate_report),
    'duplicate_policy': 'keep first: train, valid, test; fail on conflicting labels',
    'python': platform.python_version(), 'scikit_learn': sklearn.__version__,
    'numpy': np.__version__, 'opencv': cv2.__version__, 'scikit_image': skimage.__version__,
    'joblib': joblib.__version__, 'feature_count': 74,
    'selected_parameters': best_model.get_params(), 'validation_trials': trials,
    'evaluation': metrics,
    'preliminary_model': True,
    'missing_evaluation_classes': {split: [label for label in CLASS_NAMES if counts[split][label] == 0] for split in ['valid', 'test']},
    'limitations': ['Linen has no validation/test images in this run.', 'Exact duplicates filtered; garment/burst grouping not verified.', 'Low held-out accuracy; review before application deployment.'],
}
(OUTPUT_DIR / 'training_report.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'duplicate_report.json').write_text(json.dumps(duplicate_report, indent=2), encoding='utf-8')
reloaded = joblib.load(model_path)
np.testing.assert_allclose(reloaded.predict_proba(X_test), best_model.predict_proba(X_test))
print('Export reload check passed:', OUTPUT_DIR)
archive = shutil.make_archive('/content/' + OUTPUT_DIR.name + '_fabricare_model', 'zip', OUTPUT_DIR)
files.download(archive)

## Compare stronger classifiers
Run after the dataset-check cell. Reuses its features, excludes detectable burst overlap between splits, selects by validation macro F1, and exports a separate preliminary model. V3 includes linen in the validation and test splits. Collect independent phone photos for external validation.


In [ ]:
"""Run after the corrected dataset-check cell; preserves the active API model."""
import re
import time
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

assert 'data' in globals() and 'manifest' in globals(), 'Run the corrected dataset-check cell first.'
assert sklearn.__version__ == '1.6.1', 'Use the backend-compatible scikit-learn version.'

# Verify each feature row has its own matching provenance record.
improvement_records = {}
for split in ['train', 'valid', 'test']:
    records = [r for r in manifest if r['split'] == split]
    x, y = data[split]
    assert len(records) == len(x) == len(y), f'{split}: provenance length mismatch'
    assert [r['label'] for r in records] == list(y), f'{split}: provenance order mismatch'
    improvement_records[split] = records

def burst_group(record):
    # A timestamped burst is a detectable photo session, not proof of garment identity.
    stem = Path(record['path']).stem
    stem = re.sub(r'\s*\(\d+\)$', '', stem)
    match = re.match(r'(?i)^(.*?)[_-]burst\d+', stem)
    return (record['label'], match.group(1).lower()) if match else None

test_groups = {g for r in improvement_records['test'] if (g := burst_group(r)) is not None}
valid_groups = {g for r in improvement_records['valid'] if (g := burst_group(r)) is not None}
improved_data, improved_manifest, burst_exclusions = {}, [], []
for split in ['train', 'valid', 'test']:
    blocked = (test_groups | valid_groups) if split == 'train' else test_groups if split == 'valid' else set()
    keep = []
    for record in improvement_records[split]:
        group = burst_group(record)
        retain = group is None or group not in blocked
        keep.append(retain)
        if retain:
            improved_manifest.append(record)
        else:
            burst_exclusions.append({**record, 'reason': 'burst group also present in a held-out split'})
    mask = np.asarray(keep, dtype=bool)
    improved_data[split] = (data[split][0][mask], data[split][1][mask])
    assert mask.any(), f'{split}: burst filtering emptied the split; collect independent photos.'
    print(split, 'retained:', int(mask.sum()), 'excluded burst overlap:', int((~mask).sum()), flush=True)

train_x, train_y = improved_data['train']
valid_x, valid_y = improved_data['valid']
assert set(train_y) == set(CLASS_NAMES), 'Burst filtering removed a training class; review the dataset.'
supported_valid_labels = sorted(set(valid_y))
print('Untested validation classes:', sorted(set(CLASS_NAMES) - set(valid_y)), flush=True)
print('Selection uses validation macro F1 over labels actually represented in validation.', flush=True)

candidate_specs = [('decision_tree_baseline', DecisionTreeClassifier(
    class_weight='balanced', max_depth=12, min_samples_leaf=10, random_state=SEED))]
for leaf in [1, 3]:
    for features in ['sqrt', 0.5]:
        candidate_specs.append((f'extra_trees_leaf{leaf}_features{features}', ExtraTreesClassifier(
            n_estimators=400, min_samples_leaf=leaf, max_features=features,
            class_weight='balanced', n_jobs=2, random_state=SEED)))
    candidate_specs.append((f'random_forest_leaf{leaf}', RandomForestClassifier(
        n_estimators=400, min_samples_leaf=leaf, max_features='sqrt',
        class_weight='balanced_subsample', n_jobs=2, random_state=SEED)))
for c in [1, 10, 100]:
    candidate_specs.append((f'scaled_rbf_svm_C{c}', make_pipeline(StandardScaler(), SVC(
        C=c, gamma='scale', class_weight='balanced', probability=True,
        random_state=SEED, cache_size=512))))

improvement_trials = []
selected_model, selected_name, selected_score = None, None, -1
baseline_model = None
for name, spec in candidate_specs:
    started = time.monotonic()
    model = clone(spec).fit(train_x, train_y)
    prediction = model.predict(valid_x)
    score = float(f1_score(valid_y, prediction, labels=supported_valid_labels, average='macro', zero_division=0))
    result = {'name': name, 'validation_macro_f1_supported_classes': score,
        'validation_accuracy': float(accuracy_score(valid_y, prediction)),
        'training_seconds': round(time.monotonic() - started, 2)}
    improvement_trials.append(result)
    print(result, flush=True)
    if name == 'decision_tree_baseline':
        baseline_model = model
    if score > selected_score:
        selected_model, selected_name, selected_score = model, name, score

print('SELECTED USING VALIDATION ONLY:', selected_name, 'macro F1:', selected_score, flush=True)

# Evaluate the chosen model and the retrained baseline once on the same test set.
# The test labels below do not participate in model selection.
test_x, test_y = improved_data['test']
supported_test_labels = sorted(set(test_y))
def evaluation(model):
    prediction = model.predict(test_x)
    return {
        'accuracy': float(accuracy_score(test_y, prediction)),
        'macro_f1_supported_classes': float(f1_score(test_y, prediction,
            labels=supported_test_labels, average='macro', zero_division=0)),
        'supported_classes': supported_test_labels,
        'missing_classes': sorted(set(CLASS_NAMES) - set(test_y)),
        'classification_report': classification_report(test_y, prediction,
            labels=CLASS_NAMES, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(test_y, prediction, labels=CLASS_NAMES).tolist(),
        'confusion_matrix_labels': CLASS_NAMES,
    }

selected_evaluation = evaluation(selected_model)
baseline_evaluation = evaluation(baseline_model)
print('TEST selected:', selected_evaluation['accuracy'], 'baseline:', baseline_evaluation['accuracy'], flush=True)
print('TEST macro F1 (supported classes):', selected_evaluation['macro_f1_supported_classes'], flush=True)
print('UNTESTED classes:', selected_evaluation['missing_classes'], flush=True)
print(classification_report(test_y, selected_model.predict(test_x), labels=CLASS_NAMES, zero_division=0), flush=True)

improvement_dir = Path('/content/drive/MyDrive/Fabricare_training_v3') / (
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_classifier_comparison')
improvement_dir.mkdir(parents=True, exist_ok=False)
joblib.dump(selected_model, improvement_dir / 'fabric_glcm_decision_tree.pkl', compress=3)
(improvement_dir / 'fabric_labels.json').write_text(json.dumps(CLASS_NAMES, indent=2))
improvement_report = {
    'selected_model': selected_name, 'seed': SEED, 'feature_count': 74,
    'selection_metric': 'validation macro F1 on supported classes',
    'validation_supported_classes': supported_valid_labels,
    'candidates': improvement_trials,
    'selected_test': selected_evaluation, 'baseline_test_same_split': baseline_evaluation,
    'counts': {split: {label: int(np.sum(y == label)) for label in CLASS_NAMES}
        for split, (x, y) in improved_data.items()},
    'excluded_burst_images': len(burst_exclusions),
    'preliminary_model': True,
    'limitations': ['Garment identity and non-burst related photos have not been verified.',
        'Linen has no validation or test images.',
        'Test split was evaluated in a previous run; collect a new independent phone/garment test set before claiming final accuracy.'],
    'versions': {'python': platform.python_version(), 'scikit_learn': sklearn.__version__,
        'numpy': np.__version__, 'opencv': cv2.__version__, 'scikit_image': skimage.__version__, 'joblib': joblib.__version__},
}
for name, payload in [('training_report.json', improvement_report),
        ('dataset_manifest.json', improved_manifest), ('burst_exclusions.json', burst_exclusions),
        ('duplicate_report.json', duplicate_report)]:
    (improvement_dir / name).write_text(json.dumps(payload, indent=2))
# Preserve computed features for future experiments without rereading all Drive photos.
np.savez_compressed(improvement_dir / 'glcm_features.npz',
    **{f'{split}_{kind}': value for split, pair in improved_data.items()
       for kind, value in zip(['x', 'y'], pair)})
loaded = joblib.load(improvement_dir / 'fabric_glcm_decision_tree.pkl')
np.testing.assert_array_equal(loaded.predict(test_x), selected_model.predict(test_x))
np.testing.assert_allclose(loaded.predict_proba(test_x), selected_model.predict_proba(test_x))
improvement_archive = shutil.make_archive('/content/' + improvement_dir.name, 'zip', improvement_dir)
print('EXPORT VERIFIED:', improvement_dir, flush=True)
print('Archive:', improvement_archive, flush=True)
from google.colab import files
files.download(improvement_archive)
